In [0]:
# COMMAND ----------
# MAGIC %run ../log/00_audit_logging

# COMMAND ----------
import os

# Ensure catalog, schema, and raw volume exist in Unity Catalog
spark.sql("CREATE CATALOG IF NOT EXISTS workspace")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.tmdb")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.tmdb.raw")

# Raw Volume Base Directory & Subdirectories
RAW_BASE = "/Volumes/workspace/tmdb/raw"
FULL_LOAD_DIR = f"{RAW_BASE}/full_load"
INCREMENTAL_DIR = f"{RAW_BASE}/incremental"

os.makedirs(FULL_LOAD_DIR, exist_ok=True)
os.makedirs(INCREMENTAL_DIR, exist_ok=True)

# Fallback logger definition in case %run didn't bind the function
if 'log_pipeline_execution' not in globals():
    def log_pipeline_execution(layer, load_type, batch_id, read_path, start_time, end_time, status, rows_inserted=0, rows_quarantined=0, error_msg=""):
        print(f"[AUDIT LOG] Layer: {layer} | Type: {load_type} | Batch: {batch_id} | Status: {status} | Inserted: {rows_inserted} | Quarantined: {rows_quarantined}")
        if error_msg:
            print(f"[AUDIT LOG ERROR] {error_msg}")

In [0]:
# COMMAND ----------
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, BooleanType, ArrayType
)

BRONZE = "workspace.tmdb.bronze_movies"
QUARANTINE = "workspace.tmdb.quarantine_movies"

# Target Bronze Delta Table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE} (
    pulled_at         STRING,
    source_endpoint   STRING,
    source_collection STRING,
    source_file       STRING,
    raw_record        STRING,
    batch_id          STRING,
    load_timestamp    TIMESTAMP
) USING DELTA
""")

# Quarantine Delta Table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {QUARANTINE} (
    layer          STRING,
    batch_id       STRING,
    source_file    STRING,
    reject_reason  STRING,
    raw_record     STRING,
    load_timestamp TIMESTAMP
) USING DELTA
""")

In [0]:
# COMMAND ----------
# Dropdown includes "all" so Cell 4 can process both folders automatically
dbutils.widgets.dropdown("load_type", "all", ["all", "full", "incremental"], "Load type")
dbutils.widgets.text("source_path", "", "Folder or single .json file (blank = auto)")
dbutils.widgets.text("batch_id", "", "Batch ID (blank = auto timestamp)")

load_type   = dbutils.widgets.get("load_type")
source_path = dbutils.widgets.get("source_path").strip()
batch_id    = dbutils.widgets.get("batch_id").strip() or datetime.now().strftime("%Y%m%d_%H%M%S")

# Dynamic file pattern matching
if source_path == "":
    if load_type == "full":
        read_path = f"{FULL_LOAD_DIR}/*.json"
    elif load_type == "incremental":
        read_path = f"{INCREMENTAL_DIR}/*.json"
    else:
        # Placeholder when "all" is selected
        read_path = f"{RAW_BASE}/*/*.json"
else:
    read_path = source_path if source_path.endswith(".json") else source_path.rstrip("/") + "/*.json"

print(f"load_type={load_type} | read_path={read_path} | batch_id={batch_id}")

# Schema Definitions
full_schema = StructType([
    StructField("pulled_at",       StringType(),  True),
    StructField("source_endpoint", StringType(),  True),
    StructField("year_range",      StringType(),  True),
    StructField("enriched",        BooleanType(), True),
    StructField("enrich_limit",    LongType(),    True),
    StructField("record_count",    LongType(),    True),
    StructField("results",         ArrayType(StringType()), True),
    StructField("_corrupt",        StringType(),  True),
])

incremental_schema = StructType([
    StructField("pulled_at", StringType(), True),
    StructField("trending_today", StructType([
        StructField("source_endpoint", StringType(), True),
        StructField("record_count",    LongType(),   True),
        StructField("results",         ArrayType(StringType()), True),
    ]), True),
    StructField("changed_movies", StructType([
        StructField("source_endpoint",        StringType(), True),
        StructField("changed_id_count",       LongType(),   True),
        StructField("resolved_record_count",  LongType(),   True),
        StructField("results",                ArrayType(StringType()), True),
    ]), True),
    StructField("_corrupt", StringType(), True),
])

In [0]:
# COMMAND ----------
start_time = datetime.now()

# Point explicitly to both full_load and incremental folders
modes_to_process = [
    ("full", f"{RAW_BASE}/full_load/*.json"),
    ("incremental", f"{RAW_BASE}/incremental/*.json")
]

try:
    for mode, path in modes_to_process:
        print(f"--- Processing mode: {mode} | path: {path} ---")
        
        reader = (spark.read
                  .option("multiLine", "true")
                  .option("mode", "PERMISSIVE")
                  .option("columnNameOfCorruptRecord", "_corrupt"))

        src_file = F.col("_metadata.file_path").alias("source_file")

        if mode == "full":
            df = reader.schema(full_schema).json(path)
            ok = df.filter("_corrupt IS NULL")
            bronze_df = ok.select(
                "pulled_at", "source_endpoint",
                F.lit("full_load").alias("source_collection"),
                src_file,
                F.explode("results").alias("raw_record"),
            )
        else:
            df = reader.schema(incremental_schema).json(path)
            ok = df.filter("_corrupt IS NULL")
            trending = ok.select(
                "pulled_at",
                F.col("trending_today.source_endpoint").alias("source_endpoint"),
                F.lit("trending").alias("source_collection"),
                src_file,
                F.explode("trending_today.results").alias("raw_record"),
            )
            changed = ok.select(
                "pulled_at",
                F.col("changed_movies.source_endpoint").alias("source_endpoint"),
                F.lit("changed").alias("source_collection"),
                src_file,
                F.explode("changed_movies.results").alias("raw_record"),
            )
            bronze_df = trending.unionByName(changed)

        # Attach batch metrics
        bronze_df = (bronze_df
                     .withColumn("batch_id", F.lit(batch_id))
                     .withColumn("load_timestamp", F.current_timestamp()))

        # Process bad/corrupted records into Quarantine
        bad_df = df.filter("_corrupt IS NOT NULL").select(
            F.lit("Raw-to-Bronze").alias("layer"),
            F.lit(batch_id).alias("batch_id"),
            src_file,
            F.lit("File does not match expected schema / invalid JSON").alias("reject_reason"),
            F.coalesce(F.col("_corrupt"), F.col("pulled_at")).alias("raw_record"),
            F.current_timestamp().alias("load_timestamp"),
        )

        # Delta Merge into Quarantine Table
        quarantine_table = DeltaTable.forName(spark, QUARANTINE)
        (quarantine_table.alias("target")
          .merge(
            bad_df.alias("source"),
            "target.layer = source.layer AND target.batch_id = source.batch_id AND target.source_file = source.source_file"
          )
          .whenMatchedUpdateAll()
          .whenNotMatchedInsertAll()
          .execute())

        # Delta Merge into Bronze Table
        bronze_table = DeltaTable.forName(spark, BRONZE)
        (bronze_table.alias("target")
          .merge(
            bronze_df.alias("source"),
            "target.batch_id = source.batch_id AND target.source_file = source.source_file AND target.raw_record = source.raw_record"
          )
          .whenMatchedUpdateAll()
          .whenNotMatchedInsertAll()
          .execute())

        rows_inserted = spark.table(BRONZE).filter(F.col("batch_id") == batch_id).count()
        rows_bad = spark.table(QUARANTINE).filter(
            (F.col("layer") == "Raw-to-Bronze") & (F.col("batch_id") == batch_id)).count()

        log_pipeline_execution("Raw-to-Bronze", mode, batch_id, path,
                               start_time, datetime.now(), "SUCCESS",
                               rows_inserted=rows_inserted, rows_quarantined=rows_bad)

except Exception as e:
    log_pipeline_execution("Raw-to-Bronze", load_type, batch_id, read_path,
                           start_time, datetime.now(), "FAILURE", error_msg=str(e))
    raise e

In [0]:
%sql
-- COMMAND ----------
-- Verify table contents and record counts
SELECT batch_id, source_collection, COUNT(*) AS record_count
FROM workspace.tmdb.bronze_movies
GROUP BY batch_id, source_collection;

In [0]:
# Read and display as an interactive UI table
df = spark.read.table("workspace.tmdb.bronze_movies")
display(df)